In [ ]:
# --- run me first ---
from pathlib import Path
import os, sys

if Path.cwd().name == 'notebooks':
    os.chdir('..') # project/notebooks -> project
ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT)) # so `from src....` imports work
print('working from:', ROOT.name)

import pandas as pd
import requests
from dotenv import load_dotenv
from src.cleaning import fill_missing_trading_days, calculate_daily_returns, normalize_volume

# Load environment variables
load_dotenv()
RAW_DIR = os.getenv("DATA_DIR_RAW", "data/raw")
PROC_DIR = os.getenv("DATA_DIR_PROCESSED", "data/processed")

# Securely load the Alpha Vantage Key[cite: 14]
ALPHA_KEY = os.getenv("ALPHAVANTAGE_API_KEY")

working from: project


In [ ]:
# --- STAGE 04 & 05: ACQUISITION AND RAW STORAGE ---
ticker = "NVDA"
print(f"Downloading {ticker} data from Alpha Vantage...")

if not ALPHA_KEY or ALPHA_KEY == "your_key_here":
    raise ValueError("Please set a valid ALPHAVANTAGE_API_KEY in your .env file!")

url = "https://www.alphavantage.co/query"
params = {
    "function": "TIME_SERIES_DAILY",
    "symbol": ticker,
    "outputsize": "compact", # 'compact' gets the last 100 days. Use 'full' for 20+ years.
    "apikey": ALPHA_KEY,
    "datatype": "json"
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
data = response.json()

# Extract the time series data from the JSON
ts_key = [k for k in data.keys() if "Time Series" in k]
if not ts_key:
    raise RuntimeError(f"API Error or Rate Limit hit. Response: {data}")

series = data[ts_key[0]]

# Convert JSON to a Pandas DataFrame
df_nvda = pd.DataFrame(series).T.rename_axis('Date').reset_index()

# Alpha Vantage columns are named '1. open', '4. close', '5. volume'. 
# We filter and rename them so our Stage 06 cleaning functions still work!
df_nvda = df_nvda[['Date', '4. close', '5. volume']].copy()
df_nvda.rename(columns={'4. close': 'Close', '5. volume': 'Volume'}, inplace=True)

# Convert strings to proper numeric and date types
df_nvda['Close'] = pd.to_numeric(df_nvda['Close'])
df_nvda['Volume'] = pd.to_numeric(df_nvda['Volume'])
df_nvda['Date'] = pd.to_datetime(df_nvda['Date'])

# Sort chronologically (Alpha Vantage returns newest first by default)
df_nvda = df_nvda.sort_values('Date').reset_index(drop=True)

# Save the raw data[cite: 14]
raw_path = f"{RAW_DIR}/{ticker}_raw.csv"
df_nvda.to_csv(raw_path, index=False)
print(f"Raw {ticker} data saved to {raw_path}")
display(df_nvda.head(3))

In [ ]:
# --- STAGE 06: PREPROCESSING AND CLEAN STORAGE ---
print("Applying cleaning and feature engineering functions...")

# 1. Handle missing trading days (holidays)[cite: 16]
df_clean = fill_missing_trading_days(df_nvda, date_col='Date')

# 2. Calculate daily percentage returns[cite: 16]
df_clean = calculate_daily_returns(df_clean, price_col='Close')

# 3. Normalize the trading volume[cite: 16]
df_clean = normalize_volume(df_clean, vol_col='Volume')

# Keep only the columns we need for modeling[cite: 16]
df_final = df_clean[['Date', 'Close', 'Daily_Return', 'Normalized_Volume']]

# Save the processed data[cite: 16]
proc_path = f"{PROC_DIR}/{ticker}_processed.csv"
df_final.to_csv(proc_path, index=False)
print(f"Processed {ticker} data safely saved to {proc_path}")
display(df_final.head())

Loading raw data from: ../data/raw/sample.csv


FileNotFoundError: [Errno 2] No such file or directory: '../data/raw/sample.csv'